# Classificação de Imagens por CNN

Este notebook foi organizado em seções separadas por arquitetura de CNN para facilitar a execução no Google Colab.

Modelos incluídos:

- DenseNet121: baseline do artigo original
- ResNet50: CNN clássica e muito citada
- ConvNeXt V2: CNN moderna
- InceptionV3: arquitetura com conceito diferente

A ideia é manter a base comum no início e deixar cada modelo em um bloco próprio.

In [ ]:
# Base comum para todos os experimentos

import os
import numpy as np
import pandas as pd
import kagglehub

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score

# Download da base
path = kagglehub.dataset_download("mahyeks/almond-varieties")
dataset_dir = os.path.join(path, "dataset")
valid_extensions = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")

classes = sorted(
    nome for nome in os.listdir(dataset_dir)
    if os.path.isdir(os.path.join(dataset_dir, nome))
)

image_paths = []
labels = []
for classe in classes:
    classe_dir = os.path.join(dataset_dir, classe)
    for nome_arquivo in sorted(os.listdir(classe_dir)):
        if nome_arquivo.lower().endswith(valid_extensions):
            image_paths.append(os.path.join(classe_dir, nome_arquivo))
            labels.append(classe)

base_df = pd.DataFrame({"image_path": image_paths, "label": labels})
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(base_df["label"])
X_train_paths, X_test_paths, y_train, y_test = train_test_split(
    base_df["image_path"],
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print("Base carregada com sucesso.")
print(base_df["label"].value_counts())

Using Colab cache for faster access to the 'almond-varieties' dataset.
Base carregada com sucesso.
label
KAPADOKYA    465
AK           401
SIRA         384
NURLU        306
Name: count, dtype: int64


## DenseNet121

Baseline do artigo original. Use este bloco para a primeira comparação.

In [ ]:
# DenseNet121 - classificação com fine-tuning leve

from tensorflow.keras.applications.densenet import DenseNet121, preprocess_input
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam


def carregar_imagens(lista_paths, target_size=(224, 224)):
    imagens = []
    for image_path in lista_paths:
        img = load_img(image_path, target_size=target_size)
        img_array = img_to_array(img)
        imagens.append(img_array)
    return preprocess_input(np.asarray(imagens, dtype=np.float32))


X_train = carregar_imagens(X_train_paths.tolist())
X_test = carregar_imagens(X_test_paths.tolist())
os.makedirs("models", exist_ok=True)

base_model = DenseNet121(weights="imagenet", include_top=False, input_shape=(224, 224, 3))
x = GlobalAveragePooling2D()(base_model.output)
x = Dropout(0.3)(x)
output = Dense(len(label_encoder.classes_), activation="softmax")(x)
model = Model(inputs=base_model.input, outputs=output)

for layer in base_model.layers:
    layer.trainable = False

model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

# Treinamento sugerido no Colab
history = model.fit(
     X_train,
     y_train,
     validation_split=0.2,
     epochs=10,
     batch_size=32,
     verbose=1,
 )

model.save("models/densenet121.keras")

# Avaliação
y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)
print(accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

Epoch 1/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 70s 1s/step - accuracy: 0.1789 - loss: 1.9589 - val_accuracy: 0.1647 - val_loss: 1.6006
Epoch 2/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 4s 117ms/step - accuracy: 0.2452 - loss: 1.7482 - val_accuracy: 0.2811 - val_loss: 1.4888
Epoch 3/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 3s 97ms/step - accuracy: 0.2975 - loss: 1.6403 - val_accuracy: 0.3534 - val_loss: 1.3746
Epoch 4/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 3s 96ms/step - accuracy: 0.3166 - loss: 1.5471 - val_accuracy: 0.4337 - val_loss: 1.2816
Epoch 5/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 3s 95ms/step - accuracy: 0.3628 - loss: 1.4456 - val_accuracy: 0.5261 - val_loss: 1.1887
Epoch 6/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 3s 101ms/step - accuracy: 0.3910 - loss: 1.3489 - val_accuracy: 0.5944 - val_loss: 1.1008
Epoch 7/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 3s 97ms/step - accuracy: 0.4271 - loss: 1.2672 - val_accuracy: 0.6506 - val_loss: 1.0303
Epoch 8/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 3s 96ms/step - accuracy: 0.4693 - loss: 1.2094 - val_accuracy: 0.6948 - 

: 

## ResNet50

CNN clássica e muito citada. Use este bloco para o segundo experimento.

In [3]:
# ResNet50 - CNN clássica e muito citada

from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam


def carregar_imagens_resnet(lista_paths, target_size=(224, 224)):
    imagens = []
    for image_path in lista_paths:
        img = load_img(image_path, target_size=target_size)
        img_array = img_to_array(img)
        imagens.append(img_array)
    return preprocess_input(np.asarray(imagens, dtype=np.float32))


X_train_resnet = carregar_imagens_resnet(X_train_paths.tolist())
X_test_resnet = carregar_imagens_resnet(X_test_paths.tolist())
os.makedirs("models", exist_ok=True)

base_model = ResNet50(weights="imagenet", include_top=False, input_shape=(224, 224, 3))
x = GlobalAveragePooling2D()(base_model.output)
x = Dropout(0.3)(x)
output = Dense(len(label_encoder.classes_), activation="softmax")(x)
model_resnet = Model(inputs=base_model.input, outputs=output)

for layer in base_model.layers:
    layer.trainable = False

model_resnet.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

# Treinamento sugerido no Colab
history = model_resnet.fit(
     X_train_resnet,
     y_train,
     validation_split=0.2,
     epochs=10,
     batch_size=32,
     verbose=1,
 )

model_resnet.save("models/resnet50.keras")

# Avaliação
y_pred = np.argmax(model_resnet.predict(X_test_resnet, verbose=0), axis=1)
print(accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

: 

: 

## ConvNeXt V2

CNN moderna. Use este bloco para o terceiro experimento.

In [ ]:
# ConvNeXt V2 - CNN moderna

# No Colab, use `timm` e `torch` para uma implementação próxima da ConvNeXt V2.
# pip install timm torch torchvision

import os
import torch
import timm
from PIL import Image
from torchvision import transforms
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
import joblib
import gc
import numpy as np


# Evita a tentativa de buscar token implícito do Hugging Face no Colab
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Usando device: {device}")

feature_extractor_convnext = timm.create_model("convnextv2_tiny", pretrained=True, num_classes=0).to(device)
feature_extractor_convnext.eval()
transform_convnext = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


def carregar_imagens_convnext(lista_paths, batch_size=16):
    """
    Carrega imagens em batches para economizar memória GPU.
    Processa 16 imagens por vez (reduz uso de VRAM).
    """
    todas_features = []
    
    print(f"Processando {len(lista_paths)} imagens em batches de {batch_size}...")
    
    for i in range(0, len(lista_paths), batch_size):
        batch_paths = lista_paths[i:i+batch_size]
        imagens_batch = []
        
        for image_path in batch_paths:
            img = Image.open(image_path).convert("RGB")
            imagens_batch.append(transform_convnext(img))
        
        batch_tensor = torch.stack(imagens_batch).to(device)
        
        with torch.no_grad():
            features_batch = feature_extractor_convnext(batch_tensor).cpu().numpy()
        
        todas_features.append(features_batch)
        
        # Limpar cache da GPU entre batches
        if device == "cuda":
            torch.cuda.empty_cache()
        gc.collect()
        
        print(f"  ✓ Processadas imagens {i+1} a {min(i+batch_size, len(lista_paths))}")
    
    return np.vstack(todas_features)


print("Carregando imagens de treino...")
X_train_convnext = carregar_imagens_convnext(X_train_paths.tolist())
print(f"Forma X_train_convnext: {X_train_convnext.shape}")

print("\nCarregando imagens de teste...")
X_test_convnext = carregar_imagens_convnext(X_test_paths.tolist())
print(f"Forma X_test_convnext: {X_test_convnext.shape}")

scaler_convnext = StandardScaler()
X_train_convnext = scaler_convnext.fit_transform(X_train_convnext)
X_test_convnext = scaler_convnext.transform(X_test_convnext)

# Treinamento sugerido no Colab
svm = SVC(kernel="linear", C=1.0)
svm.fit(X_train_convnext, y_train)
os.makedirs("models", exist_ok=True)
joblib.dump(svm, "models/convnextv2_svm.joblib")
joblib.dump(scaler_convnext, "models/convnextv2_scaler.joblib")
y_pred = svm.predict(X_test_convnext)
print(accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

Usando device: cuda


Carregando imagens de treino...
Processando 1244 imagens em batches de 16...
  ✓ Processadas imagens 1 a 16
  ✓ Processadas imagens 17 a 32
  ✓ Processadas imagens 33 a 48
  ✓ Processadas imagens 49 a 64
  ✓ Processadas imagens 65 a 80
  ✓ Processadas imagens 81 a 96
  ✓ Processadas imagens 97 a 112
  ✓ Processadas imagens 113 a 128
  ✓ Processadas imagens 129 a 144
  ✓ Processadas imagens 145 a 160
  ✓ Processadas imagens 161 a 176
  ✓ Processadas imagens 177 a 192
  ✓ Processadas imagens 193 a 208
  ✓ Processadas imagens 209 a 224
  ✓ Processadas imagens 225 a 240
  ✓ Processadas imagens 241 a 256
  ✓ Processadas imagens 257 a 272
  ✓ Processadas imagens 273 a 288
  ✓ Processadas imagens 289 a 304
  ✓ Processadas imagens 305 a 320
  ✓ Processadas imagens 321 a 336
  ✓ Processadas imagens 337 a 352
  ✓ Processadas imagens 353 a 368
  ✓ Processadas imagens 369 a 384
  ✓ Processadas imagens 385 a 400
  ✓ Processadas imagens 401 a 416
  ✓ Processadas imagens 417 a 432
  ✓ Processadas ima

: 

## InceptionV3

Arquitetura com conceito diferente. Use este bloco para o quarto experimento.

In [ ]:
# InceptionV3 - modelo de arquitetura diferente

from tensorflow.keras.applications.inception_v3 import InceptionV3, preprocess_input
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam


def carregar_imagens_inception(lista_paths, target_size=(299, 299)):
    imagens = []
    for image_path in lista_paths:
        img = load_img(image_path, target_size=target_size)
        img_array = img_to_array(img)
        imagens.append(img_array)
    return preprocess_input(np.asarray(imagens, dtype=np.float32))


X_train_inception = carregar_imagens_inception(X_train_paths.tolist())
X_test_inception = carregar_imagens_inception(X_test_paths.tolist())
os.makedirs("models", exist_ok=True)

base_model = InceptionV3(weights="imagenet", include_top=False, input_shape=(299, 299, 3))
x = GlobalAveragePooling2D()(base_model.output)
x = Dropout(0.3)(x)
output = Dense(len(label_encoder.classes_), activation="softmax")(x)
model_inception = Model(inputs=base_model.input, outputs=output)

for layer in base_model.layers:
    layer.trainable = False

model_inception.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

# Treinamento sugerido no Colab
history = model_inception.fit(
     X_train_inception,
     y_train,
     validation_split=0.2,
     epochs=50,
     batch_size=32,
     verbose=1,
 )

model_inception.save("models/inceptionv3.keras")

# Avaliação
y_pred = np.argmax(model_inception.predict(X_test_inception, verbose=0), axis=1)
print(accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

: 

: 

## Carregar e testar modelos salvos

Use estas células para carregar os modelos treinados e fazer previsões sem retreinar.

In [ ]:
# Carregar DenseNet121

from tensorflow.keras.models import load_model

model_densenet = load_model("models/densenet121.keras")

# Usar dados já preprocessados ou carregar novas imagens
y_pred_densenet = np.argmax(model_densenet.predict(X_test, verbose=0), axis=1)
print("DenseNet121:")
print(f"Acurácia: {accuracy_score(y_test, y_pred_densenet):.4f}")
print(classification_report(y_test, y_pred_densenet, target_names=label_encoder.classes_))

NameError: name 'np' is not defined

In [ ]:
# Carregar ResNet50

model_resnet = load_model("models/resnet50.keras")

y_pred_resnet = np.argmax(model_resnet.predict(X_test_resnet, verbose=0), axis=1)
print("ResNet50:")
print(f"Acurácia: {accuracy_score(y_test, y_pred_resnet):.4f}")
print(classification_report(y_test, y_pred_resnet, target_names=label_encoder.classes_))

NameError: name 'np' is not defined

In [ ]:
# Carregar ConvNeXt V2

import joblib

scaler_convnext_loaded = joblib.load("models/convnextv2_scaler.joblib")
svm_convnext_loaded = joblib.load("models/convnextv2_svm.joblib")

X_test_convnext_scaled = scaler_convnext_loaded.transform(X_test_convnext)
y_pred_convnext = svm_convnext_loaded.predict(X_test_convnext_scaled)
print("ConvNeXt V2:")
print(f"Acurácia: {accuracy_score(y_test, y_pred_convnext):.4f}")
print(classification_report(y_test, y_pred_convnext, target_names=label_encoder.classes_))

NameError: name 'X_test_convnext' is not defined

In [ ]:
# Carregar InceptionV3

model_inception = load_model("models/inceptionv3.keras")

y_pred_inception = np.argmax(model_inception.predict(X_test_inception, verbose=0), axis=1)
print("InceptionV3:")
print(f"Acurácia: {accuracy_score(y_test, y_pred_inception):.4f}")
print(classification_report(y_test, y_pred_inception, target_names=label_encoder.classes_))

NameError: name 'X_test_inception' is not defined

In [ ]:
# Comparação resumida

resultados_carregados = {
    "DenseNet121": accuracy_score(y_test, y_pred_densenet),
    "ResNet50": accuracy_score(y_test, y_pred_resnet),
    "ConvNeXt V2": accuracy_score(y_test, y_pred_convnext),
    "InceptionV3": accuracy_score(y_test, y_pred_inception),
}

print("\n=== RESUMO DE RESULTADOS ===")
for modelo, acc in resultados_carregados.items():
    print(f"{modelo:20s}: {acc:.4f}")

NameError: name 'accuracy_score' is not defined